In [ ]:
import librosa
from os import path

SOURCE_PATH = path.normpath("../evaluations")
TARGET_PATH = path.normpath("../evaluations/evaluations")

# Least audio duration
AUDIO_MIN_DURATION = 2 # seconds
AUDIO_SAMPLE_RATE = 48_000

# How many frames to skip in each samples
CQT_HOP_LENGTH = 512

# Bin numbers based on octaves for input
CQT_OCTAVES = 6
CQT_BINS_PER_OCTAVE = 36

# Frame size for input uses the least frames
CQT_FEATURE_FRAMES = round(AUDIO_MIN_DURATION * AUDIO_SAMPLE_RATE / CQT_HOP_LENGTH)

# Starts from note
CQT_FMIN = librosa.note_to_hz('C1')

print({
  "audio min duration": AUDIO_MIN_DURATION,
  "feature bins": CQT_BINS_PER_OCTAVE * CQT_OCTAVES,
  "feature frames": CQT_FEATURE_FRAMES,
  "feature start Hz": CQT_FMIN,
})

{'audio min duration': 2, 'feature bins': 216, 'feature frames': 188, 'feature start Hz': 32.70319566257483}


In [5]:
from os import listdir, makedirs, path
from tqdm.notebook import tqdm
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

def target_npz_path():
    filename = f"{path.basename(TARGET_PATH)}.npz"
    return path.join(path.dirname(TARGET_PATH), filename)

def preprocess(cqt):
    """Truncates frames and zero pads features to frame length"""
    if cqt.shape[1] < CQT_FEATURE_FRAMES:
        pad_width = CQT_FEATURE_FRAMES - cqt.shape[1]
        cqt = np.pad(cqt, pad_width=((0, 0), (0, pad_width)), mode="constant")
    else:
        cqt = cqt[:, :CQT_FEATURE_FRAMES]
    return cqt

def extract_cqt_features(audio_path):
    """Extracts CQT features from an audio file."""
    try:
        y, sr = librosa.load(audio_path, sr=AUDIO_SAMPLE_RATE)
        cqt = librosa.cqt(
            y=y,
            sr=sr,
            fmin=CQT_FMIN,
            n_bins=CQT_BINS_PER_OCTAVE * CQT_OCTAVES,
            bins_per_octave=CQT_BINS_PER_OCTAVE,
            hop_length=CQT_HOP_LENGTH,
            window='hann',
        )
        cqt = np.abs(cqt)
        cqt = librosa.amplitude_to_db(cqt, ref=np.max)
        return preprocess(cqt)
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

makedirs(path.dirname(TARGET_PATH), exist_ok=True)

features = []
labels = []

for class_name in tqdm(listdir(SOURCE_PATH), desc=f"Processing dataset"):
    class_path = path.join(SOURCE_PATH, class_name)
    if not path.isdir(class_path):
        continue

    for audio_file in tqdm(listdir(class_path), desc=f"Processing {class_name}"):
        audio_path = path.join(class_path, audio_file)
        if not path.isfile(audio_path):
            continue

        cqt = extract_cqt_features(audio_path)
        if cqt is not None:
            features.append(cqt)
            labels.append(class_name)

npz_path = target_npz_path()
features = np.array(features)
labels = np.array(labels)
np.savez(npz_path, features=features, labels=labels)
print(f"Shape of CQT features: {features.shape}")
print(f"Shape of CQT labels: {labels.shape}")
print(f"Features and labels saved to {npz_path}")

Processing dataset:   0%|          | 0/36 [00:00<?, ?it/s]

Processing A#_diminished_4:   0%|          | 0/20 [00:00<?, ?it/s]

Processing A_diminished_4:   0%|          | 0/20 [00:00<?, ?it/s]

Processing A#_major_4:   0%|          | 0/20 [00:00<?, ?it/s]

Processing A_major_4:   0%|          | 0/20 [00:00<?, ?it/s]

...

Processing G#_major_4:   0%|          | 0/20 [00:00<?, ?it/s]

Processing G_major_4:   0%|          | 0/20 [00:00<?, ?it/s]

Processing G#_minor_4:   0%|          | 0/20 [00:00<?, ?it/s]

Processing G_minor_4:   0%|          | 0/20 [00:00<?, ?it/s]

Shape of CQT features: (720, 216, 188)
Shape of CQT labels: (720,)
Features and labels saved to ../evaluations.npz
